# IDM-VTON on Google Colab

Runs the open-source [IDM-VTON](https://github.com/yisol/IDM-VTON) virtual try-on model via a Gradio web UI.

**Before you start:**
- `Runtime > Change runtime type` → select a **GPU** (T4 works but is tight; Colab Pro with an A100/L4 High-RAM runtime is more reliable for this model).
- Run the cells in order, top to bottom. Don't skip cells.
- This version has all fixes from testing baked in: pinned dependency versions, the Starlette compatibility fix, and notes on which download warnings are safe to ignore.

In [1]:
# 1. Confirm you have a GPU attached
!nvidia-smi

'nvidia-smi' is not recognized as an internal or external command,
operable program or batch file.


In [2]:
# 2. Clone the Gradio-ready fork (dev branch has the app.py demo pre-wired)
%cd /content
!git clone -b dev https://github.com/camenduru/IDM-VTON-hf
%cd /content/IDM-VTON-hf

[WinError 2] The system cannot find the file specified: '/content'
c:\Users\CS\Downloads


C:\Users\CS\AppData\Roaming\Python\Python310\site-packages\IPython\core\magics\osm.py:393: UserWarning: This is now an optional IPython functionality, using bookmarks requires you to install the `pickleshare` library.
  bkms = self.shell.db.get('bookmarks', {})


[WinError 3] The system cannot find the path specified: '/content/IDM-VTON-hf'
c:\Users\CS\Downloads


Cloning into 'IDM-VTON-hf'...
Updating files:  10% (96/943)
Updating files:  11% (104/943)
Updating files:  12% (114/943)
Updating files:  13% (123/943)
Updating files:  14% (133/943)
Updating files:  15% (142/943)
Updating files:  16% (151/943)
Updating files:  17% (161/943)
Updating files:  18% (170/943)
Updating files:  19% (180/943)
Updating files:  20% (189/943)
Updating files:  21% (199/943)
Updating files:  22% (208/943)
Updating files:  23% (217/943)
Updating files:  24% (227/943)
Updating files:  25% (236/943)
Updating files:  26% (246/943)
Updating files:  27% (255/943)
Updating files:  28% (265/943)
Updating files:  29% (274/943)
Updating files:  30% (283/943)
Updating files:  31% (293/943)
Updating files:  32% (302/943)
Updating files:  33% (312/943)
Updating files:  34% (321/943)
Updating files:  35% (331/943)
Updating files:  36% (340/943)
Updating files:  37% (349/943)
Updating files:  38% (359/943)
Updating files:  39% (368/943)
Updating files:  40% (378/943)
Updating f

In [3]:
# 3. Download the auxiliary checkpoints (densepose, human parsing, openpose)
# The main IDM-VTON diffusion weights are pulled automatically from Hugging Face at runtime.
#
# NOTE ON WARNINGS: aria2 opens up to 16 parallel connections per file against Hugging Face's
# temporary signed download URLs. Some of those URLs expire mid-transfer and get rejected with
# HTTP 403 -- this is normal and aria2 automatically retries and resumes. Only worry if the final
# "Download Results" line for a file does NOT say (OK). If you see (OK) for all three files below,
# the downloads succeeded even though the log looks alarming.
!apt -y install -qq aria2

!aria2c --console-log-level=warn -c -x 16 -s 16 -k 1M \
  https://huggingface.co/camenduru/IDM-VTON/resolve/main/densepose/model_final_162be9.pkl \
  -d /content/IDM-VTON-hf/ckpt/densepose -o model_final_162be9.pkl

!aria2c --console-log-level=warn -c -x 16 -s 16 -k 1M \
  https://huggingface.co/camenduru/IDM-VTON/resolve/main/humanparsing/parsing_atr.onnx \
  -d /content/IDM-VTON-hf/ckpt/humanparsing -o parsing_atr.onnx

!aria2c --console-log-level=warn -c -x 16 -s 16 -k 1M \
  https://huggingface.co/camenduru/IDM-VTON/resolve/main/humanparsing/parsing_lip.onnx \
  -d /content/IDM-VTON-hf/ckpt/humanparsing -o parsing_lip.onnx

!aria2c --console-log-level=warn -c -x 16 -s 16 -k 1M \
  https://huggingface.co/camenduru/IDM-VTON/resolve/main/openpose/ckpts/body_pose_model.pth \
  -d /content/IDM-VTON-hf/ckpt/openpose/ckpts -o body_pose_model.pth

# Sanity check: confirm all four files actually landed on disk with nonzero size
!ls -la /content/IDM-VTON-hf/ckpt/densepose/model_final_162be9.pkl \
        /content/IDM-VTON-hf/ckpt/humanparsing/parsing_atr.onnx \
        /content/IDM-VTON-hf/ckpt/humanparsing/parsing_lip.onnx \
        /content/IDM-VTON-hf/ckpt/openpose/ckpts/body_pose_model.pth

'apt' is not recognized as an internal or external command,
operable program or batch file.
'aria2c' is not recognized as an internal or external command,
operable program or batch file.
'aria2c' is not recognized as an internal or external command,
operable program or batch file.
'aria2c' is not recognized as an internal or external command,
operable program or batch file.
'aria2c' is not recognized as an internal or external command,
operable program or batch file.
'ls' is not recognized as an internal or external command,
operable program or batch file.


In [4]:
# 4. Install dependencies with versions known to avoid conflicts on current Colab images
!pip install -q diffusers==0.25.0 transformers==4.36.2 accelerate==0.25.0 huggingface_hub==0.24.6
!pip install -q einops==0.7.0 onnxruntime cloudpickle omegaconf gradio==4.24.0
!pip install -q fvcore iopath av config peft==0.6.2

# Gradio 4.24.0 calls Starlette's older TemplateResponse(name, context) signature.
# Colab's current image resolves a newer Starlette (1.x+) that dropped support for it,
# which throws `TypeError: unhashable type: 'dict'` on every page load. Pin Starlette back down.
!pip install -q "starlette<1.0.0" "fastapi<1.0.0"

# Pydantic 2.11+ changed how it emits JSON schemas (additionalProperties can now be a plain
# bool instead of a nested object). Gradio 4.24.0's bundled gradio_client code doesn't handle
# that and crashes with `TypeError: argument of type 'bool' is not iterable` while building the
# app's API docs on startup. Pin Pydantic back down to avoid it.
!pip install -q "pydantic<2.11"

# NOTE: pip may print dependency-conflict WARNINGS here (e.g. mentioning python-fasthtml or
# google-adk wanting a newer starlette/pydantic). These are pre-installed Colab packages unrelated
# to IDM-VTON -- safe to ignore as long as pip says the pinned installs succeeded.

# 4b. Reduce VRAM usage for T4-class GPUs
#
# app.py hardcodes the internal generation resolution to 768x1024 regardless of your input image
# size -- this is the actual VRAM hog that causes CUDA OOM on a 15GB T4. This patches it down to
# 576x768 (same 3:4 aspect ratio, SDXL-friendly dimensions), cutting the latent size by ~44%.
# Safe to re-run -- it's a no-op if already patched.
!sed -i 's/(768,1024)/(576,768)/g; s/height=1024,/height=768,/; s/width=768,/width=576,/' /content/IDM-VTON-hf/app.py
!grep -n "576,768\|height=768,\|width=576," /content/IDM-VTON-hf/app.py

# Also reduce memory fragmentation
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 5.38.0 requires huggingface-hub>=0.28.1, but you have huggingface-hub 0.24.6 which is incompatible.

[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip
  DEPRECATION: Building 'antlr4-python3-runtime' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'antlr4-python3-runtime'. Discussion can be found at https://github.com/pypa/pip/issues/6334
ERROR: pip's dependency resolver does not currently take into account all the packages that are install

In [5]:
# 4c. Fix late-stage OOM during VAE decode (common when generation gets close to 100% then OOMs)
#
# The denoising loop itself is memory-efficient, but converting the final latent back into a
# pixel image (VAE decode) needs a large memory spike that isn't sliced by default -- this is
# often what tips a near-complete generation over the VRAM limit. This patch finds where the
# pipeline is assembled in app.py and enables VAE slicing/tiling plus attention slicing, which
# trade a little speed for a large reduction in that peak memory spike. Safe to re-run.
import re

path = "/content/IDM-VTON-hf/app.py"
with open(path, "r") as f:
    content = f.read()

anchor = "pipe.unet_encoder = UNet_Encoder"
if "pipe.vae.enable_slicing()" in content:
    print("Already patched.")
elif anchor not in content:
    print("WARNING: anchor line not found in app.py -- no patch applied. "
          "Run: !grep -n \"pipe = TryonPipeline\\|pipe.unet_encoder\" app.py "
          "and share the output so the patch can be adjusted.")
else:
    def insert_after(match):
        indent = re.match(r"[ \t]*", match.group(0)).group(0)
        return (match.group(0) + "\n"
                + indent + "pipe.vae.enable_slicing()\n"
                + indent + "pipe.vae.enable_tiling()\n"
                + indent + "pipe.enable_attention_slicing()")
    new_content, n = re.subn(r"^[ \t]*" + re.escape(anchor) + r".*$", insert_after, content, count=1, flags=re.MULTILINE)
    with open(path, "w") as f:
        f.write(new_content)
    print(f"Patched successfully ({n} insertion).")

FileNotFoundError: [Errno 2] No such file or directory: '/content/IDM-VTON-hf/app.py'

In [ ]:
# 5. Launch the Gradio app
# This opens a public share link -- watch the cell output for a line like:
#   Running on public URL: https://xxxxxxxxxxxx.gradio.live
# Click that link to open the try-on UI: left = person photo, next = garment photo,
# output panels on the right show the generated result.
%cd /content/IDM-VTON-hf
!python app.py

## Troubleshooting

- **HTTP 403 errors during cell 3 (checkpoint download)**: expected noise from aria2's parallel connections hitting expired signed URLs. Only a problem if the final `ls -la` sanity check at the end of cell 3 is missing a file or shows a 0-byte file -- in that case, re-run cell 3.
- **`ImportError: cannot import name 'PositionNet' from 'diffusers.models.embeddings'`**: `diffusers` version mismatch. Fixed by the pin in cell 4 -- if it recurs, `Runtime > Restart session`, then re-run cells 4 and 5 only (no need to re-clone or re-download).
- **`ERROR: pip's dependency resolver...` conflicts mentioning `python-fasthtml` or `google-adk`**: harmless. Those are unrelated packages pre-installed on Colab; as long as `starlette`, `fastapi`, and `pydantic` report successful installs, proceed.
- **`TypeError: argument of type 'bool' is not iterable` in `gradio_client/utils.py` (`get_type`)**: caused by Pydantic 2.11+ emitting a JSON schema shape that Gradio 4.24.0's bundled `gradio_client` can't parse when it builds the app's API docs on startup. Fixed by the `pydantic<2.11` pin in cell 4. If it recurs, restart the runtime and re-run cells 4 and 5.
- **`TypeError: unhashable type: 'dict'` / repeated `ERROR: Exception in ASGI application` when `app.py` starts**: known Gradio 4.24.0 vs. newer Starlette incompatibility, fixed by the pin in cell 4. If you still see it, restart the runtime and re-run cells 4 and 5. Note this error can appear even when the app is still working -- check further down the log for `Running on public URL:`; if that line is present, the try-on UI is functional despite the noise.
- **Clicking "Try-on" does nothing**: scroll all the way down in the Gradio tab to confirm you're clicking the actual generate button (not cut off in your browser view). Then check the Colab cell running `app.py` -- new log lines should appear the moment you click. If nothing appears there at all, refresh the gradio.live tab (websocket may have disconnected) and try again.
- **`torch.OutOfMemoryError: CUDA out of memory`**: the model's internal generation resolution (768x1024, hardcoded regardless of your input image size) plus the DensePose model together are tight against a T4's ~15GB VRAM. Fixed by the resolution patch in cell 4b, which drops internal generation to 576x768. If you get further into generation (e.g. ~90%) before OOMing, that's usually the VAE decode step at the end -- fixed by the slicing/tiling patch in cell 4c. You can also try lowering "Denoising Steps" in the Gradio UI's Advanced Settings (e.g. from 30 to 20) for extra headroom. If it still OOMs after all of this with a clean runtime restart, T4 genuinely doesn't have enough VRAM for this model -- Colab Pro with an A100 or L4 High-RAM runtime is the real fix at that point. After any OOM crash, always `Runtime > Restart session` before retrying -- failed attempts leave fragmented GPU memory that makes the next attempt fail even sooner.
- **Example person images error out**: use your own uploaded photos instead of the built-in examples.
- **Re-running after a restart**: only re-run cell 4 (dependencies) and cell 5 (launch) -- the cloned repo and downloaded checkpoints persist for the rest of the session.